# Binary T-x-y Diagram

This notebook demonstrates how to generate a T-x-y diagram for a binary mixture using `chemthermo`.

We calculate the bubble point (liquid) and dew point (vapor) curves at a fixed pressure.

## 1. Inputs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chemthermo as ct

# --- Mixture Definition ---
component_names = ("Ethanol", "Water")
pressure_Pa = 101325.0  # 1 atm
n_points = 25  # Grid resolution

# Standard EOS logic (using NRTL for this polar mixture if configured, otherwise PR)
eos = ct.PengRobinsonEOS()
# For robust demo with PR:
component_names = ("Propane", "n-Butane")
components = tuple(ct.Component.from_database(name) for name in component_names)

print(f"System: {component_names} at {pressure_Pa/101325.0:.2f} atm")

## 2. Compute Phase Envelope

We iterate through composition space (x1 from 0 to 1) and calculate:
- **Bubble Point**: The temperature where liquid starts to boil.
- **Dew Point**: The temperature where vapor starts to condense.

In [ ]:
x1_range = np.linspace(0.0, 1.0, n_points)

bubble_t = []
dew_t = []

print("Computing phase envelope...")

for x1 in x1_range:
    x2 = 1.0 - x1
    z = (x1, x2)
    
    # Create a Mixture object for the calculation
    mix = ct.Mixture(components=components, composition=ct.Composition(fractions=z))
    
    # Calculate Bubble Point
    # T_bub is the temperature where liquid of composition z boils.
    # The equilibrium vapor will have composition y.
    T_bub, y_eq = ct.bubble_temperature(mix, pressure_Pa, eos)
    bubble_t.append(T_bub)
    
    # Calculate Dew Point
    # T_dew is the temperature where vapor of composition z begins to condense.
    # The equilibrium liquid will have composition x.
    T_dew, x_eq = ct.dew_temperature(mix, pressure_Pa, eos)
    dew_t.append(T_dew)

print("Done.")

## 3. Plot Diagram

In [ ]:
plt.figure(figsize=(8, 6))

# Plot T vs x1 (Bubble Curve) and T vs y1 (Dew Curve)
# The Bubble Curve is the locus of (z, T_bubble).
# The Dew Curve is the locus of (z, T_dew).

plt.plot(x1_range, bubble_t, 'b.-', label='Bubble Point (Liquid)')
plt.plot(x1_range, dew_t, 'r.-', label='Dew Point (Vapor)')

plt.xlabel(f"Mole Fraction {component_names[0]}")
plt.ylabel("Temperature [K]")
plt.title(f"T-x-y Diagram: {component_names[0]} + {component_names[1]} at {pressure_Pa/1000:.1f} kPa")
plt.grid(True, alpha=0.3)
plt.legend()

plt.show()

## 4. Customization & Export

To customize this notebook:
1. Change `component_names` in Part 1 to other species in the database (e.g. `("Methane", "Ethane")`).
2. Change `pressure_Pa`.

### Export Data to CSV

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "x1": x1_range,
    "T_bubble_K": bubble_t,
    "T_dew_K": dew_t
})

print("Data Preview:")
print(df.head())

# df.to_csv("txy_data.csv", index=False)
# print("Saved to txy_data.csv")